# 01 — EDA et qualité des capteurs (Pump Sensor Data)

Objectifs :
1. Vérifier la régularité de l'index temporel.
2. Identifier les 7 pannes et leurs remises en service.
3. Établir le profil de qualité de chaque capteur : NaN %, variance, % constant, plus longue séquence figée, ADF/KPSS.
4. Lister les capteurs morts et suspects, avec leur justification.
5. Superposer la chronologie des pannes aux signaux.

Toute la logique est dans `src/`. Ce notebook ne fait qu'orchestrer et afficher. Les seuils sont dans `src/config.py`.

Prérequis : `data/raw/sensor.csv` (voir le README).

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import display

from src.config import PRE_FAILURE_WINDOW, TABLES
from src.data_loading import STATUS_COL, load_pump, pump_sensor_columns
from src.events import events_to_frame, extract_failure_events
from src.quality import classify_sensors, profile_sensors, time_index_report
from src.viz import plot_failure_heatmaps, plot_failure_timeline, plot_quality_overview, save_fig

pd.set_option("display.max_rows", 60)
pd.set_option("display.width", 200)

## 1. Chargement et index temporel

In [ ]:
df = load_pump()
sensors = pump_sensor_columns(df)
print(f"{df.shape[0]:,} lignes × {len(sensors)} capteurs")
display(time_index_report(df.index).to_frame("valeur"))
display(df[STATUS_COL].value_counts().to_frame("n_points"))

## 2. Pannes

Une panne correspond à une séquence contiguë de `BROKEN`. Son instant `t_f` est le premier point `BROKEN`.

In [ ]:
events = extract_failure_events(df[STATUS_COL])
events_table = events_to_frame(events)
events_table.to_csv(TABLES / "pannes.csv")
print(f"{len(events)} pannes détectées")
display(events_table)

## 3. Profil de qualité par capteur

- `nan_pct` : part de valeurs manquantes.
- `mode_pct` : part des mesures égales à la valeur la plus fréquente.
- `zero_step_pct` : % constant, c'est-à-dire la part des pas sans variation. Une valeur élevée indique un capteur figé ou fortement quantifié.
- `longest_frozen` : plus longue séquence de valeurs identiques.
- ADF / KPSS sur la moyenne horaire : `contradictoire` signale en général des ruptures ou des changements de régime (à exploiter en phase 3).

In [ ]:
profile = profile_sensors(df, sensors)
classified = classify_sensors(profile)
classified.to_csv(TABLES / "profil_capteurs.csv")

cols = ["status", "nan_pct", "std", "iqr", "mode_pct", "zero_step_pct", "longest_frozen",
        "adf_pvalue", "kpss_pvalue", "stationarity"]
display(classified[cols].round(3))

In [ ]:
fig = plot_quality_overview(classified)
save_fig(fig, "01_qualite_capteurs");

In [ ]:
display(classified["stationarity"].value_counts().to_frame("n_capteurs"))

## 4. Capteurs morts et suspects

- **Mort**, exclu de la suite : ≥ 50 % de NaN, variance nulle, ou une seule valeur couvrant ≥ 95 % des mesures.
- **Suspect**, conservé mais traité en phase 2 : ≥ 20 % de NaN, ou figé pendant ≥ 24 h.

In [ ]:
flagged = classified.loc[classified["status"] != "ok", ["status", "nan_pct", "mode_pct", "longest_frozen", "justification"]]
flagged.to_csv(TABLES / "capteurs_morts_suspects.csv")
display(flagged.sort_values("status"))

alive = classified.index[classified["status"] != "mort"].tolist()
print(f"{len(alive)} capteurs conservés sur {len(sensors)}")

## 5. Chronologie des pannes superposée aux signaux

- Trait rouge : panne (`t_f`).
- Bande orange : remise en service (`RECOVERING`).
- Bande grise : fenêtre pré-panne du protocole (48 h par défaut).

In [ ]:
fig = plot_failure_timeline(df, alive, events, resample="1h", window=PRE_FAILURE_WINDOW)
save_fig(fig, "01_chronologie_pannes");

Zoom sur chaque panne, de 72 h avant à 12 h après. Chaque capteur est exprimé en z-score robuste, calculé par rapport à l'ensemble des périodes `NORMAL`. Les bandes rouges ou bleues qui apparaissent **avant** `t_f = 0` désignent les capteurs à suivre en phase 5.

In [ ]:
fig = plot_failure_heatmaps(df, alive, events, before="72h", after="12h", resample="10min")
save_fig(fig, "01_zoom_pannes");

## 6. Constats (à rédiger après exécution)

- Régularité de l'index (trous, doublons) :
- Capteurs morts retenus, et pourquoi :
- Capteurs suspects et traitement prévu en phase 2 (imputation, masque, suppression) :
- Durées des remises en service, et pannes rapprochées (fenêtres pré-panne tronquées ?) :
- Capteurs qui dérivent visiblement avant les pannes (zooms) :
- Capteurs « contradictoires » ADF/KPSS, candidats à la segmentation des régimes :